In [11]:
%matplotlib inline
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Папка, где лежат отдельные xlsx-файлы по профилям (поправь если нужно)
PROFILES_DIR = Path("results_models/profiles")
assert PROFILES_DIR.exists(), f"Папка с профилями не найдена: {PROFILES_DIR}"

OUT_DIR = Path("results_models/profile_analysis")
(OUT_DIR / "plots" / "profiles").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "plots" / "sites").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "plots" / "corr").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "corr_matrices").mkdir(parents=True, exist_ok=True)

print("Reading profile files from:", PROFILES_DIR)
print("Outputs will be saved to:", OUT_DIR)

Reading profile files from: results_models\profiles
Outputs will be saved to: results_models\profile_analysis


In [12]:
def safe_str(x):
    return "" if pd.isna(x) else str(x).strip()

def clean_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(r'\s+', '', regex=True)
         .str.replace('*', '', regex=False)
         .str.replace(',', '.', regex=False)
         .str.replace('−','-', regex=False)
         .str.replace(r'[^0-9\.\-]', '', regex=True),
        errors='coerce'
    )

def infer_site_profile_from_filename(p: Path):
    name = p.stem
    parts = name.split("__")
    if len(parts) >= 2:
        site = parts[0]
        profile = "__".join(parts[1:])
    else:
        m = re.match(r"(.+?)__?ПРОФИЛЬ_?(.+)", name, flags=re.I)
        if m:
            site = m.group(1)
            profile = m.group(2)
        else:
            if "профиль" in name.lower():
                site = name.split("профиль")[0].strip("_- ")
                profile = name
            else:
                site = name
                profile = name
    return safe_str(site), safe_str(profile)

# Соберём все xlsx в папке (и подпапках)
files = list(PROFILES_DIR.rglob("*.xls*"))
print("Found files:", len(files))

rows = []
file_table = []
for f in files:
    try:
        df = pd.read_excel(f, engine=None)
    except Exception as e:
        print("Failed to read", f, ":", e)
        continue

    site, profile = infer_site_profile_from_filename(f)
    col_map = {c: safe_str(c).lower() for c in df.columns}
    date_col = None
    ipn_col = None
    igp_col = None
    rgp_col = None
    gp_col = None
    punkt_col = None
    for c in df.columns:
        low = col_map[c]
        if 'дат' in low:
            date_col = c
        if 'ипн' in low or 'бров' in low:
            ipn_col = c
        if 'игп' in low or 'измер' in low:
            igp_col = c
        if 'ргп' in low or 'пн' in low:
            rgp_col = c
        if low.strip() == 'гп' or low.strip() == 'gp':
            gp_col = c
        if 'пункт' in low:
            punkt_col = c

    if date_col is None and df.shape[1] >= 1:
        date_col = df.columns[0]

    for idx, row in df.iterrows():
        try:
            if date_col is not None:
                date_val = pd.to_datetime(row[date_col], dayfirst=True, errors='coerce')
            else:
                date_val = pd.NaT
        except Exception:
            date_val = pd.NaT

        ipn_v = clean_num_series(pd.Series([row[ipn_col]]))[0] if ipn_col in df.columns else np.nan
        igp_v = clean_num_series(pd.Series([row[igp_col]]))[0] if igp_col in df.columns else np.nan
        rgp_v = clean_num_series(pd.Series([row[rgp_col]]))[0] if rgp_col in df.columns else np.nan
        gp_v = None
        if gp_col in df.columns:
            try:
                gp_v = safe_str(row[gp_col])
            except:
                gp_v = None
        punkt_v = None
        if punkt_col in df.columns:
            punkt_v = safe_str(row[punkt_col])

        if (pd.isna(date_val) and pd.isna(ipn_v) and pd.isna(igp_v) and pd.isna(rgp_v)):
            continue

        rows.append({
            'file': str(f),
            'site': site,
            'profile': profile,
            'date_raw': row[date_col] if date_col in df.columns else None,
            'Дата': date_val,
            'Год': int(date_val.year) if not pd.isna(date_val) else np.nan,
            'Пункт': punkt_v,
            'ГП': gp_v,
            'ИПН_м': ipn_v,
            'ИГП_м': igp_v,
            'РГП_ПН_м': rgp_v
        })
    file_table.append({'file': str(f), 'site': site, 'profile': profile, 'rows': df.shape[0]})

df_all = pd.DataFrame(rows)
print("Combined rows:", len(df_all))
pd.DataFrame(file_table).to_excel(OUT_DIR / "files_table.xlsx", index=False)
df_all.to_excel(OUT_DIR / "observations_from_files.xlsx", index=False)
df_all.head()

Found files: 33
Combined rows: 680


,file,site,profile,date_raw,Дата,Год,Пункт,ГП,ИПН_м,ИГП_м,РГП_ПН_м
0,results_models\profiles\Бережновка__ПРОФИЛЬ___...,Бережновка,ПРОФИЛЬ___60,1958-10-01,1958-10-01,1958,None,136,249.42,125.40,249.42
1,results_models\profiles\Бережновка__ПРОФИЛЬ___...,Бережновка,ПРОФИЛЬ___60,1960-10-23,1960-10-23,1960,None,136,230.24,106.22,230.24
2,results_models\profiles\Бережновка__ПРОФИЛЬ___...,Бережновка,ПРОФИЛЬ___60,1965-07-31,1965-07-31,1965,None,136,198.78,74.76,198.78
3,results_models\profiles\Бережновка__ПРОФИЛЬ___...,Бережновка,ПРОФИЛЬ___60,1970-07-18,1970-07-18,1970,None,136,182.19,58.17,182.19
4,results_models\profiles\Бережновка__ПРОФИЛЬ___...,Бережновка,ПРОФИЛЬ___60,1974-07-13,1974-07-13,1974,None,135,172.92,11.80,172.92


In [14]:
# Ячейка 3 — нелинейный прогноз (полином 2-й степени) и улучшенная графика
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

def plot_profile_ipn_forecast(dfp, out_png, ycol='ИПН_м', title=None, degree=2):
    d = dfp.dropna(subset=['Год', ycol]).sort_values('Год')
    # Для полинома степени degree нужно минимум degree+1 точка, но для надёжности возьмём минимум 3
    if d.shape[0] < max(3, degree+1):
        return False
    
    X = d['Год'].values.reshape(-1, 1)
    y = d[ycol].values
    
    # Создаём модель полиномиальной регрессии
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    
    # Прогноз на 20 лет вперёд
    last_year = int(X.max())
    future_years = np.arange(last_year + 1, last_year + 21).reshape(-1, 1)
    X_all = np.vstack([X, future_years])
    y_all_pred = model.predict(X_all)
    
    # Построение графика
    plt.figure(figsize=(9, 5))
    
    # Фактические данные
    plt.scatter(X, y, color='steelblue', s=50, label='Фактические данные', zorder=3)
    plt.plot(X, y_pred, color='darkorange', linewidth=2, label=f'Полином {degree}-й степени (R² = {r2:.3f})')
    
    # Прогноз (пунктир)
    plt.plot(future_years, y_all_pred[-len(future_years):], 
             color='crimson', linestyle='--', linewidth=2, label='Прогноз на 20 лет')
    
    # Оформление
    plt.xlabel('Год', fontsize=12)
    plt.ylabel('ИПН, м', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.7)
    if title:
        plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.close()
    return True

In [15]:
def safe_filename(s: str) -> str:
    return re.sub(r'[^\w\d\-]', '_', str(s))[:120]

n_saved = 0
for (site, profile), grp in df_all.groupby(['site','profile']):
    out_name = OUT_DIR / "plots" / "profiles" / f"{safe_filename(site)}__{safe_filename(profile)}.png"
    title = f"{site} / {profile}"
    ok = plot_profile_ipn_forecast(grp, out_name, ycol='ИПН_м', title=title, degree=2)
    if ok:
        n_saved += 1
print("Profile forecast plots saved:", n_saved, "files ->", (OUT_DIR / "plots" / "profiles").resolve())

Profile forecast plots saved: 31 files -> C:\Users\user\Desktop\Diplom\po\results_models\profile_analysis\plots\profiles


In [16]:
n_site_plots = 0
if df_all.empty:
    print("Нет данных для построения site-level графиков")
else:
    site_year = df_all.groupby(['site','Год'])['ИПН_м'].mean().reset_index()
    for site, grp in site_year.groupby('site'):
        grp = grp.dropna(subset=['Год','ИПН_м']).sort_values('Год')
        if grp.shape[0] < 2:
            continue
        out_name = OUT_DIR / "plots" / "sites" / f"site_forecast_{safe_filename(site)}.png"
        ok = plot_profile_ipn_forecast(grp.rename(columns={'ИПН_м':'ИПН_м'}), out_name,
                                       ycol='ИПН_м', title=f"Участок: {site} (среднее ИПН по году)", degree=2)
        if ok:
            n_site_plots += 1
print("Site-level forecast plots saved:", n_site_plots, "->", (OUT_DIR / "plots" / "sites").resolve())

Site-level forecast plots saved: 10 -> C:\Users\user\Desktop\Diplom\po\results_models\profile_analysis\plots\sites


In [17]:
from matplotlib import cm

def save_corr_heatmap(corr_df, out_png, title=None):
    plt.figure(figsize=(10,8))
    sns.heatmap(corr_df, cmap='coolwarm', center=0, annot=True, fmt='.2f', square=True, linewidths=.5)
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

n_corr = 0
for site, g in df_all.groupby('site'):
    pivot = g.pivot_table(index='Год', columns='profile', values='ИПН_м', aggfunc='mean')
    pivot = pivot.dropna(axis=1, thresh=3)
    if pivot.shape[1] <= 1:
        continue
    corr = pivot.corr()
    corr.to_excel(OUT_DIR / "corr_matrices" / f"{safe_filename(site)}_corr_matrix.xlsx")
    out_png = OUT_DIR / "plots" / "corr" / f"{safe_filename(site)}_corr_heatmap.png"
    save_corr_heatmap(corr, out_png, title=f"Корреляции профилей — {site} (по ИПН)")
    n_corr += 1
print("Correlation matrices/heatmaps created for sites:", n_corr, "->", (OUT_DIR / "corr_matrices").resolve())

Correlation matrices/heatmaps created for sites: 10 -> C:\Users\user\Desktop\Diplom\po\results_models\profile_analysis\corr_matrices


In [18]:
df_all.to_csv(OUT_DIR / "observations_combined.csv", index=False)
summary = df_all.groupby(['site','profile']).agg(
    n_obs = ('ИПН_м','count'),
    year_first = ('Год','min'),
    year_last = ('Год','max'),
    mean_IPN = ('ИПН_м','mean'),
    std_IPN = ('ИПН_м','std')
).reset_index()
summary.to_excel(OUT_DIR / "profile_summary.xlsx", index=False)
print("Saved combined CSV and profile summary ->", OUT_DIR)
summary.head()

Saved combined CSV and profile summary -> results_models\profile_analysis


,site,profile,n_obs,year_first,year_last,mean_IPN,std_IPN
0,Бережновка,ПРОФИЛЬ___60,24,1958,2023,108.174583,63.325368
1,Бережновка,ПРОФИЛЬ___61,24,1958,2023,91.362083,77.396049
2,Бережновка,ПРОФИЛЬ___62,23,1958,2023,226.794792,99.586717
3,Бурты,ПРОФИЛЬ___1,25,1987,2023,61.444400,9.176637
4,Бурты,ПРОФИЛЬ___2,25,1987,2023,51.828400,11.180792


In [23]:
# Ячейка 8 — графики временных рядов для сильно коррелированных пар профилей
min_corr = 0.7
n_pairs = 0

def shorten_profile_name(name, max_len=15):
    """Укорачивает название профиля: убирает 'ПРОФИЛЬ___', обрезает до max_len символов."""
    name = re.sub(r'ПРОФИЛЬ[_\s]*', '', name, flags=re.I)
    name = re.sub(r'профиль[_\s]*', '', name, flags=re.I)
    if len(name) > max_len:
        name = name[:max_len] + '...'
    return name

for site, g in df_all.groupby('site'):
    # Строим сводную таблицу: годы × профили, значения ИПН
    pivot = g.pivot_table(index='Год', columns='profile', values='ИПН_м', aggfunc='mean')
    pivot = pivot.dropna(axis=1, thresh=3)  # удаляем профили с <3 наблюдениями
    if pivot.shape[1] <= 1:
        continue
    
    # Корреляционная матрица для отбора пар
    corr = pivot.corr()
    cols = corr.columns.tolist()
    
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            val = corr.iloc[i, j]
            if pd.isna(val) or abs(val) < min_corr:
                continue
            
            p1, p2 = cols[i], cols[j]
            # Выбираем данные по двум профилям, убираем пропуски
            df_pair = pivot[[p1, p2]].dropna()
            if df_pair.shape[0] < 3:
                continue
            
            # График временных рядов
            plt.figure(figsize=(10, 8))
            
            # Профиль 1 — синий
            plt.plot(df_pair.index, df_pair[p1], 'o-', color='steelblue', linewidth=2, markersize=6,
                     label=f'{shorten_profile_name(p1)} (синий)')
            # Профиль 2 — коричневый
            plt.plot(df_pair.index, df_pair[p2], 's-', color='saddlebrown', linewidth=2, markersize=6,
                     label=f'{shorten_profile_name(p2)} (коричневый)')
            
            plt.xlabel('Год', fontsize=12)
            plt.ylabel('ИПН, м', fontsize=12)
            plt.title(f'{site}: Сравнение профилей {shorten_profile_name(p1)} и {shorten_profile_name(p2)} (r={val:.2f})',
                      fontsize=14, fontweight='bold')
            plt.grid(True, linestyle=':', alpha=0.7)
            plt.legend(fontsize=11)
            plt.tight_layout()
            
            fn = OUT_DIR / "plots" / "corr" / f"{safe_filename(site)}__timeseries_{safe_filename(p1)}__{safe_filename(p2)}.png"
            plt.savefig(fn, dpi=150, bbox_inches='tight')
            plt.close()
            n_pairs += 1

print("Saved time series plots for strongly correlated profile pairs:", n_pairs)

C:\Users\user\AppData\Local\Temp\ipykernel_14380\1606525495.py:51: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(fontsize=11)
C:\Users\user\AppData\Local\Temp\ipykernel_14380\1606525495.py:51: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(fontsize=11)


Saved time series plots for strongly correlated profile pairs: 32


In [27]:
# Ячейка 9 — дополнительные импорты для LSTM/GRU
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [28]:
# Ячейка 10 — вспомогательные функции для LSTM/GRU

def create_sequences(data, n_steps):
    """Преобразует временной ряд в обучающие пары (X, y) с окном n_steps."""
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps])
    return np.array(X), np.array(y)

def recursive_forecast(model, last_sequence, n_future, scaler):
    """
    Рекурсивно прогнозирует на n_future шагов вперёд.
    last_sequence: исходное окно (нормализованное) формы (n_steps, 1)
    Возвращает массив прогнозов в исходном масштабе.
    """
    current_seq = last_sequence.copy().reshape(1, -1, 1)
    forecasts = []
    for _ in range(n_future):
        next_pred = model.predict(current_seq, verbose=0)[0, 0]
        forecasts.append(next_pred)
        # Обновляем окно: убираем первый элемент, добавляем предсказание
        current_seq = np.roll(current_seq, -1, axis=1)
        current_seq[0, -1, 0] = next_pred
    # Обратное масштабирование
    forecasts = np.array(forecasts).reshape(-1, 1)
    return scaler.inverse_transform(forecasts).flatten()

def train_lstm_gru_for_profile(profile_data, n_steps=5, n_future=20, test_size=0.2):
    """
    profile_data: DataFrame с колонками ['Год', 'ИПН_м'] для одного профиля.
    Возвращает словарь с результатами.
    """
    df = profile_data.dropna(subset=['ИПН_м']).sort_values('Год').copy()
    if len(df) < n_steps + 5:
        return None  # недостаточно данных
    
    years = df['Год'].values
    values = df['ИПН_м'].values.reshape(-1, 1)
    
    # Масштабирование
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    
    # Создание последовательностей
    X, y = create_sequences(scaled.flatten(), n_steps)
    
    # Разделение на train/test с сохранением временного порядка
    split = int(len(X) * (1 - test_size))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    
    # Изменение формы для LSTM/GRU: (samples, timesteps, features)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
    
    # Создание моделей
    models = {}
    history = {}
    
    # LSTM
    lstm_model = Sequential([
        LSTM(50, activation='relu', return_sequences=True, input_shape=(n_steps, 1)),
        Dropout(0.2),
        LSTM(50, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')
    
    # GRU
    gru_model = Sequential([
        GRU(50, activation='relu', return_sequences=True, input_shape=(n_steps, 1)),
        Dropout(0.2),
        GRU(50, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    gru_model.compile(optimizer='adam', loss='mse')
    
    # Ранняя остановка
    es = EarlyStopping(monitor='loss', patience=20, verbose=0, restore_best_weights=True)
    
    # Обучение LSTM
    lstm_history = lstm_model.fit(X_train, y_train, epochs=200, batch_size=4, 
                                   validation_data=(X_test, y_test), 
                                   callbacks=[es], verbose=0)
    models['LSTM'] = lstm_model
    history['LSTM'] = lstm_history
    
    # Обучение GRU
    gru_history = gru_model.fit(X_train, y_train, epochs=200, batch_size=4,
                                 validation_data=(X_test, y_test),
                                 callbacks=[es], verbose=0)
    models['GRU'] = gru_model
    history['GRU'] = gru_history
    
    # Оценка на тесте
    metrics = {}
    for name, model in models.items():
        y_pred_scaled = model.predict(X_test, verbose=0)
        y_pred = scaler.inverse_transform(y_pred_scaled)
        y_true = scaler.inverse_transform(y_test.reshape(-1, 1))
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        metrics[name] = rmse
    
    # Рекурсивный прогноз на n_future лет вперёд
    last_window = scaled[-n_steps:].flatten()  # последние n_steps значений
    future_years = np.arange(years[-1] + 1, years[-1] + n_future + 1)
    
    forecasts = {}
    for name, model in models.items():
        fore = recursive_forecast(model, last_window, n_future, scaler)
        forecasts[name] = fore
    
    return {
        'years_hist': years,
        'values_hist': values.flatten(),
        'future_years': future_years,
        'forecasts': forecasts,
        'metrics': metrics,
        'scaler': scaler,
        'models': models,
        'history': history
    }

In [32]:
# Ячейка 11 — запуск обучения для каждого профиля и сохранение графиков
n_steps = 10          # размер окна (сколько прошлых лет используем для прогноза)
n_future = 20        # горизонт прогноза (лет)
min_points = 10      # минимальное число точек для обучения

results = {}

# Проходим по всем профилям в df_all
for (site, profile), grp in df_all.groupby(['site', 'profile']):
    print(f"Обработка: {site} / {profile}")
    profile_data = grp[['Год', 'ИПН_м']].dropna().sort_values('Год')
    if len(profile_data) < min_points:
        print(f"  Пропущено: недостаточно данных ({len(profile_data)} < {min_points})")
        continue
    
    res = train_lstm_gru_for_profile(profile_data, n_steps=n_steps, n_future=n_future)
    if res is None:
        print(f"  Пропущено: ошибка подготовки")
        continue
    
    results[(site, profile)] = res
    
    # Построение графика
    plt.figure(figsize=(12, 6))
    
    # Исторические данные
    plt.plot(res['years_hist'], res['values_hist'], 'o-', color='black', label='Факт', linewidth=1.5, markersize=4)
    
    # Прогноз LSTM
    plt.plot(res['future_years'], res['forecasts']['LSTM'], 's--', color='blue', label=f"LSTM (RMSE={res['metrics']['LSTM']:.2f})", linewidth=2)
    
    # Прогноз GRU
    plt.plot(res['future_years'], res['forecasts']['GRU'], '^--', color='green', label=f"GRU (RMSE={res['metrics']['GRU']:.2f})", linewidth=2)
    
    # Линейный тренд для сравнения (опционально)
    z = np.polyfit(res['years_hist'], res['values_hist'], 1)
    p_lin = np.poly1d(z)
    years_all = np.concatenate([res['years_hist'], res['future_years']])
    plt.plot(years_all, p_lin(years_all), ':', color='red', label=f'Линейный тренд', linewidth=1.5, alpha=0.7)
    
    plt.xlabel('Год', fontsize=12)
    plt.ylabel('ИПН, м', fontsize=12)
    plt.title(f"{site} / {profile} — Сравнение прогнозов LSTM и GRU", fontsize=14, fontweight='bold')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(fontsize=10)
    plt.tight_layout()
    
    # Сохранение
    fname = safe_filename(f"{site}__{profile}__lstm_gru_forecast.png")
    out_path = OUT_DIR / "plots" / "profiles" / fname
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  График сохранён: {out_path}")

print(f"\nВсего обработано профилей: {len(results)}")

Обработка: Бережновка / ПРОФИЛЬ___60
  График сохранён: results_models\profile_analysis\plots\profiles\Бережновка__ПРОФИЛЬ___60__lstm_gru_forecast_png
Обработка: Бережновка / ПРОФИЛЬ___61
  График сохранён: results_models\profile_analysis\plots\profiles\Бережновка__ПРОФИЛЬ___61__lstm_gru_forecast_png
Обработка: Бережновка / ПРОФИЛЬ___62
  График сохранён: results_models\profile_analysis\plots\profiles\Бережновка__ПРОФИЛЬ___62__lstm_gru_forecast_png
Обработка: Бурты / ПРОФИЛЬ___1
  График сохранён: results_models\profile_analysis\plots\profiles\Бурты__ПРОФИЛЬ___1__lstm_gru_forecast_png
Обработка: Бурты / ПРОФИЛЬ___2
  График сохранён: results_models\profile_analysis\plots\profiles\Бурты__ПРОФИЛЬ___2__lstm_gru_forecast_png
Обработка: Бурты / ПРОФИЛЬ___3
  График сохранён: results_models\profile_analysis\plots\profiles\Бурты__ПРОФИЛЬ___3__lstm_gru_forecast_png
Обработка: Молчановка / ПРОФИЛЬ___57
  График сохранён: results_models\profile_analysis\plots\profiles\Молчановка__ПРОФИЛЬ___57__l

KeyboardInterrupt: 